# topology_learning usage

This notebook shows the first learning-oriented interface for satellite topology selection.

The module does not rebuild motif, region, or LST constraints. Those are produced by deterministic modules first. Here we consume metric CSVs and export selector outputs or supervised teacher datasets.

Current paper1 convention: region groups are metric endpoint sets only; they do not force region-internal +grid links. LST is applied after a schedule is selected.

## 1. Current full-link gap selector

This self-contained cell runs the current no-region-+grid dynamic selector. The objective is closeness to `full_link` on shortest delay and shortest hops, plus a penalty for newly added edges when switching topology.

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.topology_learning.module.full_link_gap_selector import run_full_link_gap_selector_from_yaml

CONFIG = GENERIC_ROOT / "src" / "topology_learning" / "examples" / "configs" / "g60_w4h3_full_link_gap_china_europe_hop_delay.yaml"

# max_rows=20 is a quick smoke run; remove it for the full 0..86160 stride-60 run.
out_dir = run_full_link_gap_selector_from_yaml(CONFIG, max_rows=20)
out_dir

## Legacy A. Export supervised dataset from YAML

This legacy cell is self-contained. It reads the old `region_internal_plus_grid` YAML config, builds the teacher dataset, and writes `dataset.npz`, `teacher_by_step.csv`, `action_summary.csv`, and `meta.json`. Keep it for old-result reproducibility; do not treat it as the current paper1 convention.

In [ ]:
import sys
from pathlib import Path

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.topology_learning.module.runner import run_topology_learning_dataset_from_yaml

CONFIG = GENERIC_ROOT / "src" / "topology_learning" / "examples" / "configs" / "g60_w4h3_region_internal_plus_grid_china_europe.yaml"

out_dir = run_topology_learning_dataset_from_yaml(CONFIG, max_rows=20)
out_dir

## Legacy B. Inspect the NPZ

`action_valid_mask` is important: it is the legal-action mask that a future GNN policy must use before selecting an action.

In [ ]:
import numpy as np
from pathlib import Path

DATASET = Path(r"E:\paper11\data\satnet_experiments\runs\paper1\G60\topology_learning\g60_w4h3_region_internal_plus_grid_china_europe_selector\dataset.npz")
data = np.load(DATASET, allow_pickle=False)

print(data.files)
print("steps", data["steps"].shape)
print("actions", data["topology_names"].shape)
print("action_values", data["action_values"].shape)
print("action_valid_mask", data["action_valid_mask"].shape)
print("teacher classes", len(np.unique(data["labels"])))
print("first labels", data["labels"][:10])

## Legacy C. Run the dependency-free baseline

This is only an interface smoke test. It uses time features, so it is not expected to solve the full topology-selection problem.

In [ ]:
import sys
from pathlib import Path
import numpy as np

GENERIC_ROOT = Path(r"E:\paper11\generic")
if str(GENERIC_ROOT) not in sys.path:
    sys.path.insert(0, str(GENERIC_ROOT))

from src.topology_learning.module.policies import NearestNeighborPolicy, classification_report, action_value_report

DATASET = Path(r"E:\paper11\data\satnet_experiments\runs\paper1\G60\topology_learning\g60_w4h3_region_internal_plus_grid_china_europe_selector\dataset.npz")
data = np.load(DATASET, allow_pickle=False)

features = data["sample_features"].astype("float32")
labels = data["labels"].astype("int32")
values = data["action_values"].astype("float64")

split = int(round(0.7 * len(labels)))
policy = NearestNeighborPolicy(k=5).fit(features[:split], labels[:split])
pred = policy.predict(features[split:])

report = {
    **classification_report(labels[split:], pred),
    **action_value_report(action_values=values[split:], y_true=labels[split:], y_pred=pred),
}
report

## 4. Future GNN input shape

`graph_state` already returns numpy arrays that can later be wrapped by PyTorch/PyG: `node_features`, `edge_index`, and `edge_features`. Hard constraints should still be enforced outside the neural model.